## feature-country.ipynb
Builds a sparse one-hot country feature matrix for albums, using the
`country_id_imputed` column from `sql_feature_artist_country_fast.parquet`.
Follows the same index-alignment contract as all feature notebooks (keyed on
`album_ids.pkl` from `01-album-artist-index.ipynb`).

In [1]:
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'
PARQUET_PATH = f'{DATA_DIR}/sql_feature_artist_country_fast.parquet'
ALBUM_PARQUET = f'{DATA_DIR}/mb_album.parquet'
ALBUM_ARTISTS = f'{DATA_DIR}/mb_album_artists.parquet'

In [2]:
# Load master album index (established by 01-album-artist-index.ipynb)
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)

album_index = pd.Index(album_ids)
n_albums = len(album_index)
print(f'Master album universe: {n_albums:,} albums')

Master album universe: 1,008,102 albums


In [3]:
# Load artist country data — use country_id_imputed preferentially,
# falling back to country_id where imputation was not needed.
country_df = pd.read_parquet(PARQUET_PATH, columns=['artist_id', 'country_id', 'country_id_imputed'])

country_df['country_final'] = country_df['country_id_imputed'].fillna(country_df['country_id'])
country_df = country_df.dropna(subset=['country_final'])
country_df['country_final'] = country_df['country_final'].astype(int)

print(f'Artists with a country signal: {len(country_df):,}')
print(f'Unique countries: {country_df["country_final"].nunique():,}')

Artists with a country signal: 1,780,246
Unique countries: 11,068


In [4]:
# Join artist country onto albums via mb_album_artists
album_artists = pd.read_parquet(ALBUM_ARTISTS, columns=['album_id', 'artist_id']).drop_duplicates()

album_country = (
    album_artists
    .merge(country_df[['artist_id', 'country_final']], on='artist_id', how='inner')
    .drop_duplicates(subset='album_id')   # one country per album (primary artist)
)

print(f'Albums with a country signal: {len(album_country):,}')

Albums with a country signal: 1,952,046


In [5]:
# Build sparse one-hot country matrix aligned to master album index
country_codes  = pd.Categorical(album_country['country_final'])
n_countries    = len(country_codes.categories)

row_idx = album_index.get_indexer(album_country['album_id'].values)
valid   = row_idx >= 0

X_country = csr_matrix(
    (np.ones(valid.sum(), dtype=np.float32),
     (row_idx[valid], country_codes.codes[valid])),
    shape=(n_albums, n_countries)
)

print(f'X_country shape : {X_country.shape}')
print(f'Non-zero entries: {X_country.nnz:,}')

X_country shape : (1008102, 2263)
Non-zero entries: 883,503


In [6]:
save_npz(f'{FEATURES_DIR}/album_country_matrix.npz', X_country)
print('Saved: album_country_matrix.npz')

Saved: album_country_matrix.npz
